In [ ]:
import pandas as pd
import string
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import random
import numpy as np
import math
dataset = pd.read_csv("sign_lang_mnist/sign_mnist_train/sign_mnist_train.csv")
alphabet = {index:letter for index, letter in enumerate(string.ascii_lowercase)} 
prex = dataset.drop(columns=["label"])
X = prex.to_numpy().reshape(-1,28,28,1)
y = dataset["label"].to_numpy()
print(y[3])
X_train,X_test,y_train,y_test = train_test_split(
    X,y,
    test_size=0.2,
    random_state=42,
    stratify=y
    )

# sow = [random.randint(0,len(X_train)-1) for i in range(10)]

# for i in sow:
#     img = X_train[i]
#     label = y_train[i]
#     plt.figure(figsize=(50,50))

#     plt.subplot(28, 28, 1)        
#     plt.imshow(img, cmap=plt.cm.gray)
#     plt.title(f"alphabet {alphabet.get(label,'unknown')}")

In [ ]:
max_pooled_activations = []
activations = []
pre_activations = []
beta_1 = 0.90
beta_2 = 0.999
learning_rate = 0.001
moment_1_history = {}
moment_2_history = {}

bias_moment_1_history = {}
bias_moment_2_history = {}

epsilon = 1e-8

In [ ]:
def softmax(pre_activations):
    # axis=-1 safely handles both 1D vectors and 2D batches
    # Subtracting in-place or inline saves memory overhead
    exps = np.exp(pre_activations - np.max(pre_activations, axis=-1, keepdims=True))
    
    return exps / np.sum(exps, axis=-1, keepdims=True)

In [ ]:
def sparse_categorical_crossentropy_loss(predictions, true_classes):
    eps = 1e-15
    preds  = np.clip(predictions,eps,1-eps)
    indexes = true_classes.index
    loss = -np.mean(np.log(preds[np.arange(len(true_classes)),true_classes]))
    return loss


In [ ]:
def accuracy(batch_predictions, true_classes):
    preds = np.asarray(batch_predictions)
    targets = np.asarray(true_classes)
    
    predicted_classes = np.argmax(preds, axis=1)
    matches = (predicted_classes ==  targets)
    accuracy = np.mean(matches)

    print(f"Predicted: {predicted_classes}")
    print(f"Matches:   {matches}")
    print(f"Accuracy:  {accuracy:.2%}")
    

In [ ]:
def softmax_sparse_categorical_crossentropy_gradient(logits_prediction_arr, true_class_idx):
    grad = logits_prediction_arr.copy()
    for i in range(len(logits_prediction_arr)):
        grad[i][true_class_idx[i]]-=1
    return grad

In [ ]:
def relu(bias_added_weights):
    return np.where(bias_added_weights>0,bias_added_weights,0)

In [ ]:
def relu_derivative(pre_activations):
    return (pre_activations>0).astype(float)

In [ ]:
def adam_optimizer(curr_weights, curr_gradients, iter, layer, is_bias=False):

    if is_bias:
        moment_1_curr = (beta_1*(bias_moment_1_history.get(layer,np.zeros(curr_weights.shape)))) + (1-beta_1) * curr_gradients
        bias_moment_1_history[layer] = moment_1_curr
    else:
        moment_1_curr = (beta_1*(moment_1_history.get(layer,np.zeros(curr_weights.shape)))) + (1-beta_1) * curr_gradients
        moment_1_history[layer] = moment_1_curr

    hat_moment_1 = moment_1_curr/(1-beta_1**iter)

    if is_bias:
        moment_2_curr = beta_2*(bias_moment_2_history.get(layer,np.zeros(curr_weights.shape))) + (1-beta_2) * curr_gradients**2
        bias_moment_2_history[layer] = moment_2_curr
    else:
        moment_2_curr = beta_2*(moment_2_history.get(layer,np.zeros(curr_weights.shape))) + (1-beta_2) * curr_gradients**2
        moment_2_history[layer] = moment_2_curr

    hat_moment_2 = moment_2_curr/(1-beta_2**iter)

    return ((learning_rate/(np.sqrt(hat_moment_2)+epsilon)) * hat_moment_1)

In [ ]:
def max_pooling(activations_matrices):
    stride_row,strice_col=(2,2)
    num_samples, num_activations, act_height, act_width = activations_matrices.shape
    max_pooled_height = act_height//2
    max_pooled_width = act_width//2
    batch_max_pooled_activations = []
    for sampl in range(num_samples):
        max_pooled_activations = np.zeros(shape=(num_activations,max_pooled_height,max_pooled_width))
        for dp in range(num_activations):
            last_row = act_height-stride_row
            last_col = act_width-strice_col
            curr_row = 0
            curr_col = 0
            while curr_row<=last_row:
                while curr_col<=last_col:
                    max_pooled_act = max_pooled_activations[dp,:,:]
                    max_pooled_h = int(math.floor(curr_row/2))
                    max_pooled_w = int(math.floor(curr_col/2))
                    # print("wthelly",sampl,dp,curr_row,curr_col,max_pooled_h,max_pooled_w)
                    max_pooled_act[max_pooled_h,max_pooled_w] = np.max(
                        activations_matrices[sampl,dp,curr_row:curr_row+2,curr_col:curr_col+2]
                        )
                    curr_col+=2
                curr_col=0
                curr_row+=2
        batch_max_pooled_activations.append(np.array(max_pooled_activations))
    return np.array(batch_max_pooled_activations)

In [ ]:
def flatten_channel_last(activations):
    new_activations = np.array(activations).copy()
    act_h,act_w,act_d = new_activations.shape
    flattened_activations = []
    for i in range(act_h):
        for j in range(act_w):
            flattened_activations.append(new_activations[i,j,:].to_list())
    return np.array(flattened_activations)

In [ ]:
def adam_optimizer():
    pass

In [ ]:
def convolve(layer_type,filter_kernels,layer_biases,batch_inputs,stride=1):
    num_inputs = len(batch_inputs)
    _,_,_,num_filters = filter_kernels.shape
    all_activations = []
    for inp in batch_inputs:
        inp_act = []
        for flt in range(num_filters):
            curr_filter = filter_kernels[:,:,:,flt]
            filter_height, filter_width,filter_depth = curr_filter.shape
            input_height,input_width,input_depth = tuple([s for s in inp.shape] + [1]) if len(inp.shape) < 3 else inp.shape
            for dp in range(input_depth):
                last_row = input_height-filter_height
                last_col = input_width-filter_width
                curr_row = 0
                curr_col = 0
                activation = np.zeros(shape=(last_row+1,last_col+1))
                while curr_row<=last_row:
                    while curr_col<=last_col:
                        activation[curr_row][curr_col] = np.sum( 
                            inp[curr_row:curr_row+filter_width,curr_col:curr_col+filter_width,dp] 
                            * curr_filter[:,:,dp]  
                        ) 
                        curr_col+=stride
                    curr_col=0
                    curr_row+=stride
                activation+=layer_biases[flt]
                inp_act.append(activation)
        all_activations.append(inp_act)
    return np.asarray(all_activations)

In [ ]:
def dense_process(weights,biases,prev_activations):
    new_pre_activations = (prev_activations@weights)+biases
    return new_pre_activations

In [ ]:
def forward_propagation(layer_types,layer_weights,layer_biases,batch_inputs,batch_true_labels):
    prev_activation = np.array(batch_inputs)
    preds=None
    for i,lt in enumerate(layer_types):
        if lt in ["conv1","conv2"]:
            pre_act = convolve(lt,layer_weights[i],layer_biases[i],prev_activation)
            pre_activations.append(pre_act)
            prev_activation = relu(pre_act)
            activations.append(prev_activation)
            max_pooled = max_pooling(prev_activation)
            prev_activation = max_pooled
            max_pooled_activations.append(max_pooled)
        else:
            if layer_types[i-1]=="conv2":
                prev_activation = flatten_channel_last(prev_activation)
            if lt=="dense1":
                pre_act = dense_process(layer_weights[i],layer_biases[i],prev_activation)
                pre_activations.append(pre_act)
                prev_activation = relu(pre_act)
                activations.append(prev_activation)
            elif lt=="dense2":
                pre_act = dense_process(layer_weights[i],layer_biases[i],prev_activation)
                pre_activations.append(pre_act)
                preds = softmax(pre_act)
                activations.append(preds)
    
    return preds



In [ ]:
def back_propagation(preds):
    pass

In [ ]:
def xavier_initialization(layer_shapes, layer_weights=[],layer_biases=[]):
    rng = np.random.default_rng()
    for i in range(len(layer_shapes)):
        if len(layer_shapes[i]) <4:
            n_in,n_out = layer_shapes[i]
            biases = np.zeros((1,n_out))
        else:
            kw, kh, in_channels, out_channels = layer_shapes[i]
            receptive_field_size = kw*kh
            n_in = in_channels*receptive_field_size
            n_out = out_channels*receptive_field_size
            biases = np.zeros((1,out_channels))

        limit = np.sqrt(6/(n_in+n_out))
        weight = rng.uniform(-limit,limit,size=layer_shapes[i])
        layer_weights.append(weight)
        layer_biases.append(biases)
    return layer_weights,layer_biases


In [ ]:
def main_loop():
    layer_shapes = [(3,3,1,32),(3,3,32,64),(5*5*64,64),(64,26)]
    layer_types = ["conv1","conv2","dense1","dense2"]
    
    layer_weights = []
    layer_biases = []
    layer_weights,layer_biases = xavier_initialization(layer_shapes,layer_weights,layer_biases)
    
    epochs = 5
    batch_size = 32
    global X_train, y_train

    iters_per_epoch = int(math.ceil(len(X_train)/batch_size))
    for i in range(epochs):
        batch_iter = 0
        for j in range(iters_per_epoch):
            batch_inputs = X_train[batch_iter:batch_iter+batch_size]
            batch_true_labels = y_train[batch_iter:batch_iter+batch_size]
            preds = forward_propagation(layer_types,layer_weights,layer_biases,batch_inputs,batch_true_labels)
            back_propagation(preds)
            batch_iter+=batch_size


    
main_loop()

In [ ]:
def test_model():
    pass

In [57]:
import numpy as np
from numpy.lib.stride_tricks import sliding_window_view
np_arre = np.array([
    [i for i in range(j*10,(j+1)*10)] for j in range(10)
])
# print(np_arre)
windows = sliding_window_view(np_arre, window_shape=(2,2), axis=(0,1))
# print(windows)
kernel = np.array([[1,2],[3,4]])
output = np.einsum("ijkl,kl->ij", windows,kernel)
print(output)

[[ 76  86  96 106 116 126 136 146 156]
 [176 186 196 206 216 226 236 246 256]
 [276 286 296 306 316 326 336 346 356]
 [376 386 396 406 416 426 436 446 456]
 [476 486 496 506 516 526 536 546 556]
 [576 586 596 606 616 626 636 646 656]
 [676 686 696 706 716 726 736 746 756]
 [776 786 796 806 816 826 836 846 856]
 [876 886 896 906 916 926 936 946 956]]
